# 02 - Unity Catalog governance (FreshMart stockout)

Governs the 17 tables published by pipeline `ef370b68-6c02-4d8e-94b4-426403033591` into `serverless_stable_zlgeaw_catalog.stockout_core`.

| Control | How | Where |
| --- | --- | --- |
| Table and column comments | `comment=` / `schema=` in pipeline code | pipeline source (source of truth) |
| PII column mask | UC SQL UDF `stockout_core.mask_pii`, attached via `MASK` in the `dim_store` schema | function: this notebook; binding: pipeline |
| Tags | `domain`, `layer`, `project`, `contains_pii`; column tags `pii`, `pii_type` | this notebook |
| Grants | `account users` read gold + dims; only `stockout_pii_readers` see PII in clear | this notebook |
| Lineage | `system.access.table_lineage` raw volume -> bronze -> silver -> gold | this notebook |

The mask is defined in pipeline code (not `ALTER TABLE`) because pipeline-managed tables are re-declared on each update.

In [0]:
%sql
-- Mask: clear text only for members of stockout_pii_readers. Bound to dim_store.store_manager/manager_email/manager_phone in the pipeline schema.
CREATE OR REPLACE FUNCTION serverless_stable_zlgeaw_catalog.stockout_core.mask_pii(val STRING)
RETURNS STRING
COMMENT 'Column mask for store-manager PII: returns the value only to members of the stockout_pii_readers group, otherwise a redacted token.'
RETURN CASE WHEN is_account_group_member('stockout_pii_readers') THEN val ELSE '***REDACTED***' END;

In [0]:
C = "serverless_stable_zlgeaw_catalog.stockout_core"
types = {r.table_name: r.table_type for r in spark.sql(
    "SELECT table_name, table_type FROM serverless_stable_zlgeaw_catalog.information_schema.tables WHERE table_schema = 'stockout_core'").collect()}
kw = {"MATERIALIZED_VIEW": "MATERIALIZED VIEW", "STREAMING_TABLE": "STREAMING TABLE"}
layer = lambda t: "bronze" if t.startswith("bronze") else "gold" if t.startswith("gold") else "silver"
for t, ty in types.items():
    spark.sql(f"ALTER {kw.get(ty, 'TABLE')} {C}.{t} SET TAGS ('domain'='supply_chain', 'layer'='{layer(t)}', "
              f"'use_case'='freshmart_stockout', 'contains_pii'='{str(t == 'dim_store').lower()}')")
for col, typ in {"store_manager": "name", "manager_email": "email", "manager_phone": "phone"}.items():
    spark.sql(f"ALTER MATERIALIZED VIEW {C}.dim_store ALTER COLUMN {col} SET TAGS ('pii_type'='{typ}')")
# ML outputs (managed tables only; synced pg_* tables are managed by Lakebase)
ML = "serverless_stable_zlgeaw_catalog.stockout_ml"
for t in ["stockout_features", "stockout_predictions", "replenishment_recommendations"]:
    spark.sql(f"ALTER TABLE {ML}.{t} SET TAGS ('domain'='supply_chain', 'layer'='gold', 'use_case'='freshmart_stockout', 'contains_pii'='false')")
# Governed tag policies in this workspace: 'project' and 'pii' only allow preset values, so use 'use_case' and 'pii_type'.
print(f"tagged {len(types)} core tables + 3 ML tables + 3 PII columns")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7475732609765358>, line 7
      5 layer = lambda t: "bronze" if t.startswith("bronze") else "gold" if t.startswith("gold") else "silver"
      6 for t, ty in types.items():
----> 7     spark.sql(f"ALTER {kw.get(ty, 'TABLE')} {C}.{t} SET TAGS ('domain'='supply_chain', 'layer'='{layer(t)}', "
      8               f"'use_case'='freshmart_stockout', 'contains_pii'='{str(t == 'dim_store').lower()}')")
      9 for col, typ in {"store_manager": "name", "manager_email": "email", "manager_phone": "phone"}.items():
     10     spark.sql(f"ALTER MATERIALIZED VIEW {C}.dim_store ALTER COLUMN {col} SET TAGS ('pii_type'='{typ}')")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:919, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    916         _views.append(SubqueryAlias(df._plan, name))
    91

In [0]:
%sql
-- Analysts / app / Genie consumers: read gold + dimensions. PII stays masked unless in stockout_pii_readers.
GRANT USE CATALOG ON CATALOG serverless_stable_zlgeaw_catalog TO `account users`;
GRANT USE SCHEMA ON SCHEMA serverless_stable_zlgeaw_catalog.stockout_core TO `account users`;
GRANT SELECT ON TABLE serverless_stable_zlgeaw_catalog.stockout_core.gold_store_sku_daily TO `account users`;
GRANT SELECT ON TABLE serverless_stable_zlgeaw_catalog.stockout_core.gold_supplier_performance TO `account users`;
GRANT SELECT ON TABLE serverless_stable_zlgeaw_catalog.stockout_core.gold_stockout_kpis_daily TO `account users`;
GRANT SELECT ON TABLE serverless_stable_zlgeaw_catalog.stockout_core.dim_store TO `account users`;
GRANT SELECT ON TABLE serverless_stable_zlgeaw_catalog.stockout_core.dim_product TO `account users`;
GRANT SELECT ON TABLE serverless_stable_zlgeaw_catalog.stockout_core.dim_supplier TO `account users`;
GRANT EXECUTE ON FUNCTION serverless_stable_zlgeaw_catalog.stockout_core.mask_pii TO `account users`;

In [0]:
%sql
SELECT 'mask' kind, table_name, column_name, mask_name AS detail FROM serverless_stable_zlgeaw_catalog.information_schema.column_masks WHERE table_schema = 'stockout_core'
UNION ALL SELECT 'column_tag', table_name, column_name, concat(tag_name, '=', tag_value) FROM serverless_stable_zlgeaw_catalog.information_schema.column_tags WHERE schema_name = 'stockout_core'
UNION ALL SELECT 'table_tag', table_name, NULL, concat(tag_name, '=', tag_value) FROM serverless_stable_zlgeaw_catalog.information_schema.table_tags WHERE schema_name = 'stockout_core' AND tag_name = 'layer'
UNION ALL SELECT 'grant', table_name, grantee, privilege_type FROM serverless_stable_zlgeaw_catalog.information_schema.table_privileges WHERE table_schema = 'stockout_core' AND grantee = 'account users'
ORDER BY kind, table_name

kind,table_name,column_name,detail
grant,dim_product,account users,SELECT
grant,dim_store,account users,SELECT
grant,dim_supplier,account users,SELECT
grant,gold_stockout_kpis_daily,account users,SELECT
grant,gold_store_sku_daily,account users,SELECT
grant,gold_supplier_performance,account users,SELECT
mask,dim_store,manager_phone,serverless_stable_zlgeaw_catalog.stockout_core.mask_pii
mask,dim_store,store_manager,serverless_stable_zlgeaw_catalog.stockout_core.mask_pii
mask,dim_store,manager_email,serverless_stable_zlgeaw_catalog.stockout_core.mask_pii


In [0]:
%sql
SELECT store_id, region, store_manager, manager_email, manager_phone FROM serverless_stable_zlgeaw_catalog.stockout_core.dim_store ORDER BY store_id LIMIT 5

store_id,region,store_manager,manager_email,manager_phone
S001,West,***REDACTED***,***REDACTED***,***REDACTED***
S002,Midwest,***REDACTED***,***REDACTED***,***REDACTED***
S003,Northeast,***REDACTED***,***REDACTED***,***REDACTED***
S004,West,***REDACTED***,***REDACTED***,***REDACTED***
S005,Southwest,***REDACTED***,***REDACTED***,***REDACTED***


In [0]:
%sql
SELECT DISTINCT coalesce(source_table_full_name, source_path) AS source, target_table_full_name AS target
FROM system.access.table_lineage
WHERE target_table_catalog = 'serverless_stable_zlgeaw_catalog' AND target_table_schema = 'stockout_core'
  AND event_date >= '2026-10-25'
ORDER BY target, source

source,target
null,serverless_stable_zlgeaw_catalog.stockout_core.bronze_inventory
null,serverless_stable_zlgeaw_catalog.stockout_core.bronze_pos_sales
null,serverless_stable_zlgeaw_catalog.stockout_core.bronze_products
null,serverless_stable_zlgeaw_catalog.stockout_core.bronze_promotions
null,serverless_stable_zlgeaw_catalog.stockout_core.bronze_receipts
null,serverless_stable_zlgeaw_catalog.stockout_core.bronze_stores
null,serverless_stable_zlgeaw_catalog.stockout_core.bronze_suppliers
serverless_stable_zlgeaw_catalog.stockout_core.bronze_products,serverless_stable_zlgeaw_catalog.stockout_core.dim_product
serverless_stable_zlgeaw_catalog.stockout_core.bronze_stores,serverless_stable_zlgeaw_catalog.stockout_core.dim_store
serverless_stable_zlgeaw_catalog.stockout_core.bronze_suppliers,serverless_stable_zlgeaw_catalog.stockout_core.dim_supplier
